# Chern++ - Positive Chern Classes in Thom Polynomials

*A computational companion.*

This notebook is a tour of what the package in `src/chernpp` can do, in the order a reader of the
project's three reports would want it.

**Setting.** For the Morin singularity $A_d$, Bérczi–Szenes express the Thom polynomial as an
iterated residue whose only $d$-dependent input is $\mathcal{Q}_d$, the $T_d$-equivariant
multidegree of a Borel orbit closure $\mathcal{O}_d \subset \widehat{N}_d$. Normalising
$z_d = 1$ and setting $x_j = z_j/z_{j+1}$, the positivity question is read off the **chamber
series**

$$F_d(x) \;=\; \frac{\prod_{m<l}(1 - z_m/z_l)\,\mathcal{Q}_d}{\prod (1 - z_m/z_l - z_r/z_l)}
\;=\; \frac{N_d(x)}{\prod_r (1 - f_r(x))} \;=\; \sum_{\beta \ge 0} A_\beta x^\beta ,$$

where every $f_r$ has nonnegative coefficients and zero constant term.

Two conjectures live on top of this:

| | statement | status |
|---|---|---|
| **Rimányi**, weak | the Chern coefficients of $\mathrm{Tp}_{A_d}$ are $\ge 0$ | the real target |
| **Bérczi–Szenes**, strong | $A_\beta \ge 0$ for all $\beta$ | true for $d=4$, **false** for $d=5$ |

The strong one implies the weak one, because each Chern coefficient is a *sum* of several
$A_\beta$. The gap between them is where all the interest is.

Sections 1–6 reproduce results from `papers/`; sections 7–9 are new.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / "chernpp").is_dir()
                       else pathlib.Path.cwd() / "src"))

from chernpp.artifacts import load_algebra
from chernpp.chern import chern_coefficients, thom_polynomial
from chernpp.chamber import (ballot_orderings, chamber_series, chern_coefficient, monomial,
                             paired_defects, sorted_negatives, unpaired_tail_defects)
from chernpp.certificates import minimum_order, search_certificate
from chernpp.chamber import tail_target, tail_target_factored
from chernpp import lemma1
from chernpp.polynomial import is_nonneg, poly_to_string
from chernpp.tables import algebra_report, series_report, table
from chernpp.lorentzian import check_strong_log_concavity, extract_log_concavity_sequence

import logging; logging.disable(logging.INFO)   # keep the notebook quiet

for d in (4, 5, 6):
    a = load_algebra(d)
    print(f"A_{d}: chamber vars {a.chamber_vars}, {len(a.denominator_factors):2d} denominator factors, "
          f"numerator has {len(a.numerator):5d} terms")

A_4: chamber vars ('a', 'b', 'c'),  7 denominator factors, numerator has    54 terms
A_5: chamber vars ('a', 'b', 'c', 'e'), 13 denominator factors, numerator has   814 terms
A_6: chamber vars ('a', 'b', 'c', 'd', 'e'), 22 denominator factors, numerator has 26618 terms


## 1. The rational function

Everything downstream is generated from one pickled artifact per $d$, produced by the SageMath
miner. Nothing is hard-coded: the denominator factors below are exactly the
$1 - z_m/z_l - z_r/z_l$ of the residue formula, rewritten in the chamber variables.

In [2]:
print(algebra_report())

     vars  dim N_d  deg Q_d  Q_d terms  N_d terms  deg N_d  N_d neg  neg %  max |coeff|
---  ----  -------  -------  ---------  ---------  -------  -------  -----  -----------
A_4     3        7        1          3         54       13       27  50%              3
A_5     4       13        3         19        814       31      414  51%             18
A_6     5       22        7        418      26618       66    13309  50%            800


The growth is the point. $\mathcal{Q}_d$ goes from 3 terms to 418, and the assembled numerator
$N_d$ from 54 to 26618 with about half its coefficients negative at every order. Positivity is
asking for cancellation on that scale, which is why sampling is no substitute for a proof.

In [3]:
a5 = load_algebra(5)
print("A_5 denominator factors  1 - f_r :")
for r, f in enumerate(a5.denominator_factors):
    print(f"   f_{r:2d} = {poly_to_string(f, a5.chamber_vars)}")

A_5 denominator factors  1 - f_r :
   f_ 0 = 2*a
   f_ 1 = 2*a*b
   f_ 2 = b + a*b
   f_ 3 = 2*a*b*c
   f_ 4 = b*c + a*b*c
   f_ 5 = c + a*b*c
   f_ 6 = 2*b*c
   f_ 7 = 2*a*b*c*e
   f_ 8 = b*c*e + a*b*c*e
   f_ 9 = c*e + a*b*c*e
   f_10 = e + a*b*c*e
   f_11 = 2*b*c*e
   f_12 = c*e + b*c*e


## 2. Classical Thom polynomials

The first sanity check on the whole pipeline. At relative dimension $\ell = 0$ these are the
classical polynomials; note the coefficient of $c_d$ is $(d-1)!$ in each case.

Reproducing all eleven $A_6$ coefficients is the check that pins down $\mathcal{Q}_6$: they are
known independently, and no plausible error survives all of them at once.

In [4]:
for d in (4, 5, 6):
    print(f"Tp_A{d} (l=0)  =  {thom_polynomial(dim=d, l_max=0)}\n")

E0801 10:30:50.669882    1666 cuda_executor.cc:1176] [0] Failed to allocate device memory of 4.50GiB (4831838208 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:226
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:403

E0801 10:30:50.673351    1666 cuda_executor.cc:1176] [0] Failed to allocate device memory of 4.05GiB (4348654080 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:226
external/xla/xla/stream_executor/cuda/cuda_device_allocator.cc:403



Tp_A4 (l=0)  =  6*c_4 + 9*c_3 * c_1 + 2*c_2^2 + 6*c_2 * c_1^2 + 1*c_1^4



Tp_A5 (l=0)  =  24*c_5 + 38*c_4 * c_1 + 12*c_3 * c_2 + 25*c_3 * c_1^2 + 10*c_2^2 * c_1 + 10*c_2 * c_1^3 + 1*c_1^5



Tp_A6 (l=0)  =  120*c_6 + 202*c_5 * c_1 + 55*c_4 * c_2 + 17*c_3^2 + 141*c_4 * c_1^2 + 79*c_3 * c_2 * c_1 + 5*c_2^3 + 55*c_3 * c_1^3 + 30*c_2^2 * c_1^2 + 15*c_2 * c_1^4 + 1*c_1^6



## 3. Strong Laurent positivity fails at $d = 5$

$F_4$ is coefficientwise nonnegative — Theorem 1 of `rimanyi_positivity.pdf`. From $d = 5$ on it
is not, and the lowest-degree counterexample is
$A_{(1,1,2,1)} = [abc^2e]\,F_5 = -1$, i.e. the Laurent monomial $z_1z_3/z_4z_5$.

$A_6$ has the exact analogue at the same $\beta$, extended by a zero.

In [5]:
print(series_report(max_deg=11))

     terms (deg<=11)  negative  first negative   its degree  most negative  with i=0
---  ---------------  --------  ---------------  ----------  -------------  --------
A_4              237         0  --               --          --                    0
A_5              490         6  (1, 1, 2, 1)     5           -1                    0
A_6              796        10  (1, 1, 2, 1, 0)  5           -2                    2


## 4. Why that does not disprove Rimányi

Several Laurent monomials feed the same Chern monomial. Writing $x^\beta = z^\alpha$ with
$\alpha = (\beta_1, \beta_2-\beta_1, \dots, -\beta_{d-1})$, the monomial
$c_{\ell+2}^2 c_{\ell+1} c_\ell^2$ collects every ordering of the multiset $\{1,1,0,-1,-1\}$
whose partial sums stay nonnegative. The $-1$ is swamped.

This reproduces the table in §5 of `rimanyi_positivity.pdf` line by line.

In [6]:
F5 = chamber_series(5, 12)
M = (1, 1, 0, -1, -1)
print(f"{'alpha':>22}  {'beta':>14}  A_beta")
for alpha, beta in ballot_orderings(M):
    print(f"{str(alpha):>22}  {str(beta):>14}  {F5.get(beta, 0):>3}")
print(f"{'':>38}  ----\nC(M) = {chern_coefficient(F5, M, 12)}")

                 alpha            beta  A_beta
     (0, 1, -1, 1, -1)    (0, 1, 0, 1)    0
     (0, 1, 1, -1, -1)    (0, 1, 2, 1)    1
     (1, -1, 0, 1, -1)    (1, 0, 0, 1)    0
     (1, -1, 1, -1, 0)    (1, 0, 1, 0)    0
     (1, -1, 1, 0, -1)    (1, 0, 1, 1)    0
     (1, 0, -1, 1, -1)    (1, 1, 0, 1)    0
     (1, 0, 1, -1, -1)    (1, 1, 2, 1)   -1
     (1, 1, -1, -1, 0)    (1, 2, 1, 0)    2
     (1, 1, -1, 0, -1)    (1, 2, 1, 1)    2
     (1, 1, 0, -1, -1)    (1, 2, 2, 1)    6
                                        ----
C(M) = 10


## 5. The $\ell$-free reduction

Theorem 5 of `report.pdf`: the coefficient of $\prod_i c_{p_i}$ in $\mathrm{Tp}^\ell_{A_d}$
equals $C(M)$ with $M = \{p_i - (\ell+1)\}$ — it depends on $\ell$ *only* through $M$.

So "Rimányi for all $\ell$" is the single $\ell$-free statement $C(M) \ge 0$ over zero-sum
multisets, and raising $\ell$ merely admits multisets with more negative entries. The value $10$
above is therefore not special to $\ell = 0$:

In [7]:
for l in range(5):
    _, chern = chern_coefficients(dim=5, l_max=l)
    print(f"l = {l}:  coefficient of c_{{{l+2}}}^2 c_{{{l+1}}} c_{{{l}}}^2  =  {chern.get(tuple(sorted(M)), 0)}")

l = 0:  coefficient of c_{2}^2 c_{1} c_{0}^2  =  10


l = 1:  coefficient of c_{3}^2 c_{2} c_{1}^2  =  10


l = 2:  coefficient of c_{4}^2 c_{3} c_{2}^2  =  10


l = 3:  coefficient of c_{5}^2 c_{4} c_{3}^2  =  10


l = 4:  coefficient of c_{6}^2 c_{5} c_{4}^2  =  10


## 6. Verifying Rimányi's conjecture

Sweeping $\ell$ and checking every Chern coefficient. `report.pdf` reached $\ell \le 11$ for $A_5$
at roughly 48 s for the last step; here the whole sweep is a few seconds because the series is
expanded by XLA-compiled fixed-point iteration on an integer grid.

For $A_6$ this is new. The evaluator refuses to answer once coefficients outgrow `int64`, rather than returning
wrapped values that would read as spurious counterexamples.

In [8]:
for d in (5, 6):
    print(f"A_{d}:")
    for l in range(9):
        try:
            _, chern = chern_coefficients(dim=d, l_max=l)
        except OverflowError as ex:
            print(f"   l={l}: stopped, coefficients exceed int64"); break
        neg = sum(1 for v in chern.values() if v < 0)
        print(f"   l={l}: {len(chern):5d} Chern monomials, min coefficient {min(chern.values()):3d}"
              f"  -> {'PASS' if neg == 0 else f'FAIL ({neg} negative)'}")

A_5:


   l=0:     7 Chern monomials, min coefficient   1  -> PASS


   l=1:    30 Chern monomials, min coefficient   1  -> PASS


   l=2:    84 Chern monomials, min coefficient   1  -> PASS


   l=3:   192 Chern monomials, min coefficient   1  -> PASS


...

   l=2:   199 Chern monomials, min coefficient   1  -> PASS


   l=3:   532 Chern monomials, min coefficient   1  -> PASS


   l=4:  1206 Chern monomials, min coefficient   1  -> PASS


   l=5:  2432 Chern monomials, min coefficient   1  -> PASS


   l=6: stopped, coefficients exceed int64


## 7. The $A_5$ reduction strategy, tested at $d = 6$

`a5_weak_positivity_handoff.pdf` reduces the weak conjecture for $A_5$ to two statements about
the first adjacent transposition $\tau(i,j,k,l) = (j-i,j,k,l)$:

* **unpaired tail**, $i > j \Rightarrow A_{i,j,\dots} \ge 0$ — proved there (Proposition 3);
* **paired inequality**, $A_{i,j,\dots} + A_{j-i,j,\dots} \ge 0$ for $0 \le i \le j$ — open, and
  sufficient to finish the proof.

Both survive at $d = 6$. That is evidence the reduction is the right frame, not an $A_5$ accident.

In [9]:
for d, deg in ((5, 14), (6, 11)):
    F = chamber_series(d, deg)
    tail, pair = unpaired_tail_defects(F, deg), paired_defects(F, deg)
    print(f"A_{d} (degree <= {deg}):")
    print(f"   unpaired tail  i > j => A >= 0        : {'holds' if not tail else f'FAILS at {tail[0]}'}")
    print(f"   paired         A_b + A_tau(b) >= 0    : {'holds' if not pair else f'FAILS at {pair[0]}'}")

A_5 (degree <= 14):
   unpaired tail  i > j => A >= 0        : holds
   paired         A_b + A_tau(b) >= 0    : holds


A_6 (degree <= 11):
   unpaired tail  i > j => A >= 0        : holds
   paired         A_b + A_tau(b) >= 0    : holds


## 8. Prefix positivity holds at $d=5$ and fails at $d=6$

§8 of the handoff note observes that every tested coefficient of $B = F_5/(1-a)$ is nonnegative,
and §11.4 proposes proving that as a stepping stone, since
$B_{i,j,k,l} = \sum_{r \le i} A_{r,j,k,l}$ turns the paired inequality into a statement about
adjacent differences of a prefix array.

**The route does not survive at $d = 6$.** The prefix sum runs over $r \le i$, so a negative
coefficient with $i = 0$ passes through untouched — and $A_6$ has one, while $A_5$ has none.

In [10]:
for d, deg in ((5, 14), (6, 11)):
    nv = len(load_algebra(d).chamber_vars)
    F = chamber_series(d, deg)
    B = chamber_series(d, deg, extra_factors=[monomial(nv, 0)])
    at_i0 = [b for b, _ in sorted_negatives(F) if b[0] == 0]
    print(f"A_{d}: negatives of F_{d} with i = 0 : {at_i0 if at_i0 else 'none'}")
    print(f"      F_{d}/(1-a) : {'NONNEGATIVE' if is_nonneg(B) else f'{len(sorted_negatives(B))} negative'}\n")

A_5: negatives of F_5 with i = 0 : none
      F_5/(1-a) : NONNEGATIVE



A_6: negatives of F_6 with i = 0 : [(0, 2, 3, 2, 2), (0, 3, 4, 2, 2)]
      F_6/(1-a) : 2 negative



## 9. Denominator certificates

A way to *prove* coefficientwise positivity rather than sample it. If

$$N \;=\; \sum_{|S| \le k} P_S \prod_{r \in S} (1 - f_r), \qquad P_S \ge 0 \ \text{coefficientwise},$$

then $F = \sum_S P_S / \prod_{r \notin S}(1-f_r)$ is a sum of products of nonnegative series, so
$F \ge 0$. Call this an **order-$k$ certificate**. §10.3 of the handoff note reports an
order-1 search coming back infeasible.

The search is a feasibility LP: $P_\emptyset$ is not a free unknown but the remainder
$N - \sum_{S \ne \emptyset} P_S \prod_{r\in S}(1-f_r)$, so the only requirement is that it come
out nonnegative. Whatever the floating-point LP proposes, $P_\emptyset$ is then recomputed in
exact rational arithmetic and the certificate accepted only if it verifies — so a returned
certificate is a proof.

**$A_4$ admits one, of order exactly 4.**

In [11]:
a4 = load_algebra(4)
cert = search_certificate(a4.numerator, a4.denominator_factors, 3,
                          order=4, max_degree=8, varnames=a4.chamber_vars)
print("verified against the exact identity:",
      cert.is_valid(a4.numerator, a4.denominator_factors))
print(cert.summary()[:700], "...")

verified against the exact identity: True
order-4 certificate, deg <= 8, 25 nonzero coefficients
  (1-f_2):  1/24*a^2*b*c
  (1-f_0) * (1-f_1):  1/12*a^2*b^3*c^2
  (1-f_2) * (1-f_5):  1/6*a^2*b^3*c^2
  (1-f_2) * (1-f_6):  1/12*a^2*b*c
  (1-f_4) * (1-f_5):  1/8*a*b^2
  (1-f_4) * (1-f_6):  1/3*a^2*b^3*c^2
  (1-f_0) * (1-f_1) * (1-f_3):  1/12*a^2*b^3*c^2
  (1-f_0) * (1-f_2) * (1-f_3):  5/24*a*b*c^2
  (1-f_0) * (1-f_2) * (1-f_4):  1/12*a*b*c^2
  (1-f_0) * (1-f_3) * (1-f_5):  1/12*a*b^2
  (1-f_0) * (1-f_3) * (1-f_6):  1/6*a^2*b^3*c^2
  (1-f_0) * (1-f_4) * (1-f_5):  1/24*a*b^2
  (1-f_0) * (1-f_4) * (1-f_6):  1/6*a^2*b^3*c^2
  (1-f_1) * (1-f_2) * (1-f_6):  1/24*a*b*c^2
  (1-f_2) * (1-f_3) * (1-f_6):  1/8*a^2*b*c
  (1-f_2) * (1-f_4) * (1-f_6 ...


### Lower bounds on the order

A constraint on a monomial of total degree $T$ involves only the $P_S$ coefficients of degree
$\le T$, because every $\prod_{r\in S}(1-f_r)$ has constant term 1. Truncating both at $T$ is
therefore an *exact projection* of the feasible set — so if the truncated LP is infeasible,
**no certificate of that order exists at any degree**. Failure to find one becomes a theorem.

For $A_4$ the depth-2 probe is sharp: order $\le 3$ is impossible, and order 4 is achieved above.
For $A_5$ the obstruction is far deeper — even the prefix series $F_5/(1-a)$, which appears
coefficientwise nonnegative, admits no certificate of order $\le 7$.

In [12]:
rows = []
for d in (4, 5):
    alg = load_algebra(d); nv = len(alg.chamber_vars)
    for label, fs in (("F", alg.denominator_factors),
                      ("F/(1-a)", alg.denominator_factors + [monomial(nv, 0)])):
        bounds = [minimum_order(alg.numerator, fs, nv, probe_degree=T, max_order=7)
                  for T in (1, 2, 3)]
        rows.append((f"A_{d} {label}", bounds))

print(f"{'series':>14} | least unobstructed order at probe depth 1, 2, 3")
for name, b in rows:
    print(f"{name:>14} | " + "  ".join('>7' if x is None else str(x) for x in b))

        series | least unobstructed order at probe depth 1, 2, 3
         A_4 F | 3  4  4
   A_4 F/(1-a) | 3  4  4
         A_5 F | 4  7  >7
   A_5 F/(1-a) | 4  7  >7


## 10. Certifying the right object: multiplicative certificates

Sections 9's certificates target $F_d$ itself. That is *not* what the reduction in
`a5_weak_positivity_handoff.pdf` asks about, and the certificate *class* is wrong too.

Both successful proofs in the literature are **multiplicative**. Lemma 1 says
$(1-u)/(1-v)$ is coefficientwise nonnegative whenever $v - u \ge 0$, and Theorem 1
($d=4$) and Proposition 3 (the unpaired tail at $d=5$) are both products of such ratios.
A product of ratios is not a finite sum $\sum_S P_S \prod_{r \in S}(1-f_r)$ with $P_S \ge 0$,
so the additive engine structurally cannot find them.

The diagnostic is sharp: the additive search is obstructed on a series whose nonnegativity
is a *published theorem*. When a method fails on a known theorem, the method is wrong.

In [13]:
# J_d(1/2, ...) controls the whole unpaired tail:  for i > j,
#     A_{i,j,k,...} = 2^{i-1} [b^j c^k ...] J_d(1/2, b, c, ...).
num, den, vn = tail_target(5)
print("A_5 tail target J_5(1/2, %s):" % ",".join(vn),
      len(num), "numerator terms,", len(den), "denominators")
print("   additive least unobstructed order:",
      minimum_order(num, den, len(vn), probe_degree=2, max_order=4),
      " <- obstructed, yet this series' positivity is Proposition 3")

A_5 tail target J_5(1/2, b,c,e): 195 numerator terms, 12 denominators


   additive least unobstructed order: None  <- obstructed, yet this series' positivity is Proposition 3


### Lemma 1 rediscovers the published proof

Given the numerator in factored form -- the Vandermonde factors are already a factorisation --
the search pairs each $(1-u)$ with a denominator it dominates, cancels whatever divides
exactly, and asks whether what is left is nonnegative.

For $d = 5$ it closes completely: **leftover numerator exactly $1$**, no denominators left.
That is Proposition 3, reconstructed from scratch.

In [14]:
facs, res, dens, vn = tail_target_factored(5)
cert = lemma1.search(facs, res, dens, len(vn), vn)
print(cert.summary())

Lemma-1 certificate: 9 paired ratios, 3 denominators cancelled exactly, 0 denominators left
   (1 - [b]) / (1 - [b])
   (1 - [1/2*b]) / (1 - [3/2*b])
   (1 - [b*c]) / (1 - [b*c])
   (1 - [1/2*b*c]) / (1 - [3/2*b*c])
   (1 - [c]) / (1 - [c + 1/2*b*c])
   (1 - [b*c*e]) / (1 - [b*c*e])
   (1 - [1/2*b*c*e]) / (1 - [3/2*b*c*e])
   (1 - [c*e]) / (1 - [c*e + 1/2*b*c*e])
   ... and 1 more
   leftover numerator: 1
   => PROVED


### $d = 6$: how far it gets

The same search does not close $d = 6$. It leaves

$$\frac{1 - cde - \tfrac32 bcde}{1 - 2cde},$$

and no available denominator $v$ dominates $u = cde + \tfrac32 bcde$, so Lemma 1 cannot
finish. Expanding, the obstruction is the family $b\,(cde)^{k}$ with negative coefficients.

In [15]:
facs6, res6, dens6, vn6 = tail_target_factored(6)
cert6 = lemma1.search(facs6, res6, dens6, len(vn6), vn6)
print(cert6.summary())

Lemma-1 certificate: 14 paired ratios, 6 denominators cancelled exactly, 1 denominators left
   (1 - [b]) / (1 - [b])
   (1 - [1/2*b]) / (1 - [3/2*b])
   (1 - [b*c]) / (1 - [b*c])
   (1 - [1/2*b*c]) / (1 - [3/2*b*c])
   (1 - [c]) / (1 - [c + 1/2*b*c])
   (1 - [b*c*d]) / (1 - [b*c*d])
   (1 - [1/2*b*c*d]) / (1 - [3/2*b*c*d])
   (1 - [c*d]) / (1 - [c*d + 1/2*b*c*d])
   ... and 6 more
   leftover numerator: 1 - c*d*e - 3/2*b*c*d*e
   => not conclusive


### Backing off: a partial reduction for $A_6$

A maximum matching is greedy in the wrong way -- it can consume a denominator the remainder
needs. Returning a small subset of ratios to the remainder fixes that: two of them suffice,
leaving **12 Lemma-1 ratios times a remainder that is nonnegative as far as we check**.

This is a reduction, not yet a theorem: the remainder's nonnegativity is checked, not proved,
and its numerator has negative coefficients, so it needs a further decomposition of its own.
The honest statement is that the $A_6$ tail now sits behind one explicit rational function
in four variables rather than behind the full five-variable series.

In [16]:
back = lemma1.search_with_backoff(facs6, res6, dens6, len(vn6), vn6,
                                  max_returned=2, probe_degree=11)
print(back.summary())
print()
print("remainder numerator is itself nonnegative?  ", is_nonneg(back.numerator))
print("additive certificate for the remainder?     ",
      minimum_order(back.numerator, back.denominators, len(vn6),
                    probe_degree=2, max_order=3))

12 Lemma-1 ratios kept, 2 returned to the remainder
remainder: 207 numerator terms over 9 denominators
returned:
   (1 - [1/2*b]) / (1 - [3/2*b])
   (1 - [1/2*b*c*d*e]) / (1 - [3/2*b*c*d*e])
remainder nonnegative to degree 11

remainder numerator is itself nonnegative?   False
additive certificate for the remainder?      None


### Generalising Lemma 1: absorption

Lemma 1 needs the numerator in the special shape $1-u$. The following criterion drops that and
lets several denominators cooperate. Split $N = N_+ - N_-$ into its sign parts, and let
$1 - W = \prod_{v \in S}(1-v)$ for a subset $S$ of the denominators. If

$$N_- \;\le\; N_+ \cdot W \qquad\text{(coefficientwise)}$$

then $N - N_+\prod_S(1-v) = N_+W - N_- \ge 0$, so $N = N_+\prod_S(1-v) + P$ with $P \ge 0$, and

$$\frac{N}{\prod_{\text{all}}(1-v)} \;=\; \frac{N_+}{\prod_{v\notin S}(1-v)} \;+\;
\frac{P}{\prod_{\text{all}}(1-v)},$$

a sum of nonnegative series. Lemma 1 is the case $N = 1-u$, $S = \{v\}$, where the condition
reads $u \le v$ — so this is a strict generalisation.

**It still does not close $A_6$.** Searching jointly over back-off subsets and absorbing subsets
finds nothing among 13090 criteria. Since $J_6(1/2,\cdot)$ *is* nonnegative as far as we compute,
a proof exists — but it needs an idea outside this class. That is a statement about the method,
not about how hard we searched.

In [17]:
print("Lemma 1 case,  (1-x)/(1-2x)      :",
      lemma1.absorbs({(0,): 1, (1,): -1}, [{(1,): 2}], 1))
print("not of the form 1-u, still fine  :",
      lemma1.absorbs({(0,): 1, (1,): 1, (2,): -2}, [{(1,): 2}], 1))

subset = lemma1.absorbing_subset(back.numerator, back.denominators, len(vn6), max_size=3)
print("\nabsorbing subset for the A_6 remainder:",
      subset if subset is not None else "none exists")

Lemma 1 case,  (1-x)/(1-2x)      : True
not of the form 1-u, still fine  : True



absorbing subset for the A_6 remainder: none exists


## 11. Positivity is not Lorentzian

One structural explanation would be that the coefficient arrays are M-convex, so that positivity
follows from Lorentzian geometry. They are not: the normalised Huh–Brändén inequality
$(a_k/\binom{n}{k})^2 \ge (a_{k-1}/\binom{n}{k-1})(a_{k+1}/\binom{n}{k+1})$ fails on prefix slices.
Whatever governs Morin positivity, it is not log-concavity.

In [18]:
grid, _ = chern_coefficients(dim=5, l_max=2)
seq = extract_log_concavity_sequence(grid, 2, 1, 1)
print("prefix slice:", seq)
print("strongly log-concave (Lorentzian):", check_strong_log_concavity(seq))

prefix slice: {1: 2, 2: 8, 3: 16, 4: 32, 5: 64, 6: 128, 7: 256, 8: 512, 9: 1024, 10: 2048, 11: 4096, 12: 8192}
strongly log-concave (Lorentzian): False


## Summary

Reproduced from `papers/`: the classical Thom polynomials; $A_{(1,1,2,1)} = -1$ and its
cancellation into $C(M)=10$; the $\ell$-free reduction; failure of log-concavity.

New here:

1. **$A_6$ is verified.** Rimányi's conjecture holds for $A_6$ at every relative dimension the
   `int64` grid reaches ($\ell \le 5$).
2. **The $A_5$ reduction generalises.** Both the unpaired tail and the paired inequality hold
   for $A_6$ in the tested range.
3. **Prefix positivity does not generalise.** $F_6/(1-a)$ has negative coefficients, because
   $A_6$ — unlike $A_5$ — has negatives with $i = 0$. The stepping stone proposed in §11.4 of the
   handoff note is unavailable at $d = 6$.
4. **An explicit certificate for $A_4$**, machine-found and exactly verified, of order exactly 4.
5. **Order lower bounds are now provable**, by exact projection of the certificate LP, upgrading
   "the search failed" to "no certificate of this order exists at any degree". This extends the
   order-1 infeasibility of §10.3 considerably: nothing of order $\le 7$ works for $A_5$.
6. **The unpaired tail is one series, and Lemma 1 proves it at $d=5$ mechanically.** Our
   $J_5(1/2,b,c,e)$ reproduces the published equation (9) coefficient for coefficient, and the
   multiplicative search closes it with leftover exactly $1$. At $d = 6$ the same search reduces
   the tail to 12 Lemma-1 ratios times one explicit remainder in four variables -- a genuine
   reduction, but not yet a proof.

6. **The unpaired tail is one series, and $d=5$ is proved mechanically.** Our $J_5(1/2,b,c,e)$
   reproduces published equation (9) coefficient for coefficient, and the multiplicative search
   closes it with leftover exactly $1$. At $d=6$ the same search reduces the tail to 12 Lemma-1
   ratios times one explicit remainder in four variables.
7. **The $d=5$ proof technique provably does not extend to $d=6$**, even after generalising
   Lemma 1 to absorption.

Open, and the natural next targets: prove the paired inequality for $A_5$; find the shape of
certificate that does work (higher order, or a genuinely different grouping); and push $A_6$
past $\ell = 5$, which needs exact arithmetic (CRT over several primes) rather than `int64`.